# 03 - Exploratory Data Analysis (EDA)
AlphaForge AI - multi-asset EDA over the cached universe: coverage & quality, price trends, return distributions, volatility clustering, cross-asset correlations, macro relationships and news sentiment.

> All heavy lifting lives in `src/analysis/eda.py` so the logic is testable and reusable by the dashboard/API. Figures land in `figures/eda/`, reports in `reports/`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.analysis import eda
from src.utils.logger import setup_logging

setup_logging()

## 1. Load the universe
Cleans every cached series (dedupe, gap-stitching) exactly like the pipeline does.

In [ ]:
universe = eda.load_universe()
for group, frames in universe.items():
    print(f'{group}: {len(frames)} symbols')

## 2. Dataset overview - coverage, missing values, outliers

In [ ]:
overview = eda.overview_table(universe)
overview

## 3. Price trends & normalized performance

In [ ]:
eda.plot_price_trends(universe, 'stocks')
eda.plot_normalized_performance(universe, 'stocks', years=5)
print('figures written to figures/eda/')

## 4. Return distributions - skew, kurtosis, Jarque-Bera
Daily equity returns are fat-tailed and skewed; this motivates bounded targets and robust scaling downstream.

In [ ]:
dist_fig, dist_stats = eda.plot_return_distributions(universe, 'stocks')
dist_stats

## 5. Volatility clustering
Rolling volatility plus autocorrelation of |returns| at lags 1-10 (large positive ACF = clustering).

In [ ]:
eda.plot_rolling_volatility(universe, window=21, group='stocks')
eda.volatility_clustering_stats(universe, 'stocks')

## 6. Stationarity (ADF)
Prices are non-stationary, returns are stationary -> model returns, not levels.

In [ ]:
eda.adf_stationarity(universe, 'stocks')

## 7. Cross-asset correlation structure
Stocks vs indices vs ETFs vs commodities vs FX - drives context features and portfolio diversification.

In [ ]:
corr, corr_fig = eda.cross_asset_correlation(universe)
corr.round(2)

## 8. Macro relationships (per target symbol)
Correlation of NVDA daily returns with macro levels and day-over-day changes.

In [ ]:
macro_tbl, macro_fig = eda.macro_relationships(universe['stocks']['NVDA'])
macro_tbl

## 9. News sentiment vs same-day returns

In [ ]:
sent_tbl, sent_fig = eda.sentiment_vs_returns('NVDA')
sent_tbl

## 10. Full report
Runs everything end-to-end and writes JSON + Markdown reports to `reports/`.

In [ ]:
report = eda.run_full_eda(symbol='NVDA')
print('figures:', len(report['figures']))
print('reports:', report['report_files'])